In [4]:
import pandas as pd
import requests
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

FDC_API_KEY = os.getenv("FDC_API_KEY")

synonyms = {
    'venison': 'deer',
    'soup/stew': 'soup',
    'tree nut': 'nuts',
    'fortified wine': 'dessert wine',
    'poultry sausage': 'turkey sausage',
    'jalapeño': 'jalapeno',
    'jícama': 'jicama',
    'marscarpone': 'mascarpone',
    'rosé': 'rose wine',
    'legume': 'lentils',
}
to_mg = {'g': 1000, 'mg': 1, 'mcg': 0.001, "ug": 0.001, "µg": 0.001}
valid_values = {
    # компоненты еды
    1004: ("Fat", "g", 78),
    1258: ("Saturated fat", "g", 20),
    1253: ("Cholesterol", "mg", 300),
    1005: ("Total carbohydrates", "g", 275),
    1093: ("Sodium", "mg", 2300),
    1079: ("Dietary Fiber", "g", 28),
    1003: ("Protein", "g", 50),
    1235: ("Added sugars", "g", 50),

    # витаминчики :)
    1106: ("Vitamin A", "mcg", 900),      
    1162: ("Vitamin C", "mg", 90),
    1087: ("Calcium", "mg", 1300),
    1089: ("Iron", "mg", 18),
    1114: ("Vitamin D", "mcg", 20),      
    1109: ("Vitamin E", "mg", 15),
    1185: ("Vitamin K", "mcg", 120),
    1165: ("Thiamin", "mg", 1.2),
    1166: ("Riboflavin", "mg", 1.3),
    1167: ("Niacin", "mg", 16),
    1175: ("Vitamin B6", "mg", 1.7),
    1190: ("Folate", "mcg", 400),          
    1178: ("Vitamin B12", "mcg", 2.4),
    1176: ("Biotin", "mcg", 30),
    1170: ("Pantothenic acid", "mg", 5),
    1091: ("Phosphorus", "mg", 1250),
    1100: ("Iodine", "mcg", 150),
    1090: ("Magnesium", "mg", 420),
    1095: ("Zinc", "mg", 11),
    1103: ("Selenium", "mcg", 55),
    1098: ("Copper", "mg", 0.9),
    1101: ("Manganese", "mg", 2.3),
    1096: ("Chromium", "mcg", 35),
    1102: ("Molybdenum", "mcg", 45),
    1088: ("Chloride", "mg", 2300),
    1092: ("Potassium", "mg", 4700),
    1180: ("Choline", "mg", 550),
}

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
    'Accept': 'application/json',
    'Content-Type': 'application/json',
}

def fetch(query, datatype, retries=5):
    key = f'{query}|{datatype}'
    if key in cache:
        return query, cache[key], 200
    status = None
    for attempt in range(retries):
        try:
            response = session.get("https://api.nal.usda.gov/fdc/v1/foods/search", params={
                'query': query,
                'dataType': datatype,
                'api_key': FDC_API_KEY,
                'pageSize': 50,
            }, timeout=20)
            status = response.status_code
            if status == 200:
                return query, response.json(), 200
        except requests.RequestException as e:
            status = type(e).__name__
        time.sleep(2 ** attempt)
    return query, None, status

def fetch_all(queries, datatype):
    responses, errors ={}, {}
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = [pool.submit(fetch, query, datatype) for query in set(queries)]
        for future in as_completed(futures):
            query, data, status = future.result()
            if status != 200:
                errors[query] = status
            else:
                responses[query] = data
                cache[f'{query}|{datatype}'] = data
    return responses, errors


def to_percent(nutrient_id, amount, unit):
    something, daily_value_unit, daily_value = valid_values[nutrient_id]
    unit = unit.lower()
    if unit not in to_mg:
        return None
    amount = amount * to_mg[unit] / to_mg[daily_value_unit]
    return round(amount / daily_value * 100, 1)

def right_food(item, data):
    item = item.lower().strip()
    datatypes = {
        'Foundation': 1,
        'SR Legacy': 1.5,
        'Survey (FNDDS)': 0.5,
        'Branded': -0.5,
    }
    needed_heads = {item, item + 's', item + 'es'}
    if item.endswith('y'):
        needed_heads.add(item[:-1] + 'ies')
    words = set(item.split())

    candidates = []
    for food in data.get('foods', []):
        desc = food.get('description', '').lower()
        parts = [part.strip() for part in desc.split(',')]

        if parts[0] in needed_heads:
            head_score = 3                                  
        elif len(words) > 1 and words <= set(parts):   # 'rose wine' -> 'Wine, rose'
            head_score = 2
        elif any(part in needed_heads or part.startswith(item + ' ') for part in parts):
            head_score = 1
        else:                                 
            continue

        quality = 0
        if 'raw' in parts or 'fresh' in parts:
            quality += 2

        if food.get('dataType') in datatypes:
            quality += datatypes[food.get('dataType')]

        for nutrient in food.get('foodNutrients', []):
            nutrient_id = nutrient.get('nutrientId')
            if nutrient_id in valid_values:
                quality += 0.1

        quality -= desc.count(',') * 0.1
        candidates.append(((head_score, quality), food))

    if not candidates:
        return None
    return max(candidates, key=lambda x: x[0])[1]
    
try:
    with open('data/ingredients_cache.json', encoding='utf-8') as file:
        cache = json.load(file)
except FileNotFoundError:
    cache = {}

session = requests.Session()

df_nutrients = pd.read_csv('data/ingredient_cols.csv', index_col='Unnamed: 0')
df_nutrients = df_nutrients.rename(columns={'0': 'ingredient'})

items = df_nutrients['ingredient'].dropna().unique()
queries = {item: synonyms.get(item, item).replace('/', ' ') for item in items}

def choose(queries):
    chosen = {}
    generic, errors = fetch_all(queries, 'Foundation,SR Legacy,Survey (FNDDS)')
    for query, data in generic.items():
        food = right_food(query, data)
        if food is not None:
            chosen[query] = food

    missing = [q for q in set(queries) if q not in chosen and q not in errors]
    branded, branded_errors = fetch_all(missing, 'Branded')
    errors.update(branded_errors)
    for query, data in branded.items():
        food = right_food(query, data)
        if food is not None:
            chosen[query] = food
    return chosen, errors

def split_variants(item):
    stop_words = {'and', 'of', 'with', 'the', 'no'}
    parts = [part.strip() for part in item.split('/') if part.strip()]
    words = [word for part in parts for word in part.split() if len(word) > 2 and word not in stop_words]
    variants = []
    for variant in parts + words:
        if variant != item and variant not in variants:
            variants.append(variant)
    return variants

chosen, errors = choose(list(queries.values()))

# не нашлось целиком -> пробуем куски по слешу, потом отдельные слова
variants = {item: split_variants(item) for item, query in queries.items() if query not in chosen}
sub_chosen, sub_errors = choose([v for vs in variants.values() for v in vs])
for item, vs in variants.items():
    for variant in vs:
        if variant in sub_chosen:
            chosen[queries[item]] = sub_chosen[variant]
            break

with open('data/ingredients_cache.json', 'w', encoding='utf-8') as file:
    json.dump(cache, file)

rows = []
not_found = []

for item, query in queries.items():
    food = chosen.get(query)
    if food is None:
        reason = f'{errors[query]}' if query in errors else 'food is None'
        not_found.append({'food': item, 'reason': reason})
        continue

    row = {'ingredient': item, 'description': food.get('description')}
    for nutrient in food.get('foodNutrients', []):
        nutrient_id = nutrient.get('nutrientId')
        if nutrient_id not in valid_values:
            continue
        name = valid_values[nutrient_id][0]
        row[name] = to_percent(
            nutrient_id, nutrient.get('value', 0), nutrient.get('unitName', '')
        )
    rows.append(row)

df_result = pd.DataFrame(rows)
df_not_found = pd.DataFrame(not_found)
df_result   

,ingredient,description,Vitamin A,Folate,Vitamin E,Choline,Total carbohydrates,Fat,Niacin,Pantothenic acid,...,Manganese,Saturated fat,Cholesterol,Vitamin D,Selenium,Vitamin K,Added sugars,Iodine,Biotin,Molybdenum
0,lima bean,"Lima beans, immature seeds, raw",1.1,8.5,2.1,7.3,7.3,1.1,9.2,4.9,...,53.0,1.0,0.0,0.0,3.3,4.7,NaN,NaN,NaN,NaN
1,créme de cacao,CACAO,NaN,NaN,NaN,NaN,14.5,51.3,NaN,NaN,...,NaN,20.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,orzo,ORZO,NaN,NaN,NaN,NaN,28.6,2.3,33.5,NaN,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,banana,"Bananas, raw",0.3,5.0,0.7,1.8,8.3,0.4,4.2,6.7,...,11.7,0.6,0.0,0.0,1.8,0.4,NaN,NaN,NaN,NaN
4,tilapia,"Fish, tilapia, raw",0.0,6.0,2.7,7.7,0.0,2.2,24.4,9.7,...,1.6,2.9,16.7,15.5,76.0,1.2,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330,amaretto,"AMARETTO, IRISH CREAM, RUM HOT CHOCOLATE BOMBE...",NaN,NaN,NaN,NaN,20.4,41.0,NaN,NaN,...,NaN,100.0,2.0,NaN,NaN,NaN,84.0,NaN,NaN,NaN
331,potato salad,"Potato salad, home-prepared",3.6,1.8,NaN,NaN,4.1,10.5,5.6,10.7,...,4.4,7.1,22.7,0.0,7.5,NaN,NaN,NaN,NaN,NaN
332,poppy,"Spices, poppy seed",0.0,20.5,11.8,1.6,10.2,53.3,5.6,6.5,...,291.7,22.6,0.0,0.0,24.5,0.0,NaN,NaN,NaN,NaN
333,kiwi,"Kiwi fruit, raw",0.4,6.5,8.7,1.4,5.1,0.6,2.3,NaN,...,NaN,0.1,0.0,0.0,0.4,33.6,NaN,NaN,NaN,NaN


In [5]:
df_merged = df_nutrients.merge(df_result, on='ingredient', how='left').drop(columns=['description'])
df_merged.to_csv('data/nutrition_facts.csv', index=False)

# df_result[df_result['ingredient'].isin(df_not_found['food'])]

In [6]:
df_not_found 

,food,reason
0,kirsch,food is None
1,frangelico,food is None
2,chartreuse,food is None
3,mezcal,food is None
4,grappa,food is None
5,campari,food is None
6,ramekin,food is None
7,calvados,food is None
8,pernod,food is None
9,"no meat, no problem",food is None
